# 00 — Colab Full Pipeline (T4 GPU, source of truth)

This notebook is the **single source of truth** for the headline numbers in the report. It is meant to be run **end-to-end on Colab with a T4 GPU**.

It does the following:

1. Clones (or pulls) the repository from GitHub.
2. Installs the pinned dependencies from `requirements-colab.txt`.
3. Pins all RNGs, enables deterministic kernels, and prints an environment fingerprint.
4. Runs `scripts/run_full_pipeline.py --full` (event-labels → leakage probe → Tier-2 multi-seed → Tier-3 ablation).
5. Copies the resulting `outputs/` to Google Drive so results survive runtime shutdown.

**Estimated wall-clock on T4:** ~25 min (Tier-2 multi-seed: 5 × 4 min; Tier-3: ~5 min).

Local CPU runs of this same code path are functionally equivalent but produce different floating-point values and so are **not** comparable to the headline numbers.

## 1. Repo + dependencies

In [ ]:
# --- Edit these three lines if you fork the repo -------------------------
REPO_URL    = 'https://github.com/bahaa1515/EECE-693-project.git'
REPO_BRANCH = 'tarek-branch'
REPO_NAME   = 'EECE-693-project'
# --------------------------------------------------------------------------
import os, subprocess, sys, shutil
from pathlib import Path

if not Path(REPO_NAME).exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL])
else:
    subprocess.check_call(['git', '-C', REPO_NAME, 'fetch', 'origin', REPO_BRANCH])
    subprocess.check_call(['git', '-C', REPO_NAME, 'checkout', REPO_BRANCH])
    subprocess.check_call(['git', '-C', REPO_NAME, 'pull', '--ff-only', 'origin', REPO_BRANCH])
%cd $REPO_NAME

In [ ]:
# Install pinned versions. The PyTorch wheel index ensures CUDA 12.1 is used.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '-r', 'requirements-colab.txt',
                       '--extra-index-url', 'https://download.pytorch.org/whl/cu121'])

In [ ]:
# Force a clean re-import: ensure the just-installed wheels are what gets loaded.
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

## 2. Determinism + environment fingerprint

In [ ]:
import os, platform, json

os.environ['PYTHONHASHSEED']        = '42'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import numpy, pandas, sklearn, torch  # noqa: E401
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

info = {
    'python'   : platform.python_version(),
    'numpy'    : numpy.__version__,
    'pandas'   : pandas.__version__,
    'sklearn'  : sklearn.__version__,
    'torch'    : torch.__version__,
    'cuda'     : torch.version.cuda,
    'gpu'      : torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'cudnn'    : torch.backends.cudnn.version() if torch.cuda.is_available() else None,
}
print(json.dumps(info, indent=2))
assert torch.cuda.is_available(), 'No GPU detected — switch runtime to T4 GPU and re-run.'

## 3. Run full pipeline

This regenerates event-onset labels, runs the leakage probe (which gates the rest of the pipeline), then trains Tier-2 across 5 seeds and Tier-3 with the 4-config ablation.

In [ ]:
!python scripts/run_full_pipeline.py --full

## 4. Inspect headline results

In [ ]:
import pandas as pd
pd.read_csv('outputs/tables/headline_results.csv')

## 5. Save outputs to Google Drive

This copies `outputs/` (results CSVs + figures) and `data/processed/` to a stable Drive folder so the run survives runtime shutdown. Re-running the cell overwrites the same folder.

In [ ]:
from google.colab import drive
import shutil, os
from pathlib import Path

drive.mount('/content/drive')
DEST = Path('/content/drive/MyDrive/EECE-693-results')
DEST.mkdir(parents=True, exist_ok=True)
for sub in ('outputs', 'data/processed'):
    src = Path(sub)
    dst = DEST / sub
    if dst.exists():
        shutil.rmtree(dst)
    if src.exists():
        shutil.copytree(src, dst)
        print('copied', src, '->', dst)
print('All artefacts saved to', DEST)